[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ElenaVillano/prope-programacion/blob/main/materiales/m07_agrupacion.ipynb)

# Propedéutico de Programación para el Análisis de datos

## EGobiernoyTP

**Verano 2026**

### Material 7. Exploración de bases de datos:  agrupa y combinar información


1. **Agrupar información** con `groupby()` y `agg()`.
2. **Combinar bases de datos** con `merge()` y distintos tipos de joins.


In [ ]:
# Librerías 
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/ElenaVillano/prope-programacion/refs/heads/main/data/datos.csv')

#### Hacemos un poco de limpieza

In [ ]:
df.columns = (
    df.columns
    .str.lower()
    .str.replace(" ", "_")
)
df.head()

In [ ]:
df.shape

### Agrupar información

Muchas veces no queremos analizar cada observación individualmente, sino **resumir información para distintos grupos**.

Por ejemplo:

> ¿Cuál es el valor promedio para cada estado?


In [ ]:
df[["state", "value"]].head()

#### `groupby()`

`groupby()` divide las observaciones de acuerdo con los valores de una variable.

Por sí solo, `groupby()` todavía no calcula nada: después debemos indicar qué operación queremos realizar dentro de cada grupo.


In [ ]:
df.groupby("state")


- Promedio por estado


In [ ]:
df.groupby("state")["value"].mean()

Podemos guardar el resultado en una nueva variable:


In [ ]:
promedio_estado = (
    df
    .groupby("state")["value"]
    .mean()
)

promedio_estado.head()


##### ¿Qué pasó con `state`?

Después de agrupar, `state` se convierte en el índice del resultado.


In [ ]:
promedio_estado.index


Podemos regresar el índice a una columna con `reset_index()`:


In [ ]:
promedio_estado = (
    df
    .groupby("state")["value"]
    .mean()
    .reset_index()
)

promedio_estado.head()


##### Otras operaciones de resumen

Además del promedio, podemos calcular otros estadísticos por grupo.


In [ ]:
df.groupby("state")["value"].min()


In [ ]:
df.groupby("state")["value"].max()


In [ ]:
df.groupby("state")["value"].median()


#### `agg()`

Cuando queremos calcular **varios estadísticos al mismo tiempo**, podemos utilizar `agg()`.


In [ ]:
resumen_estado = (
    df
    .groupby("state")["value"]
    .agg(["count", "mean", "median", "min", "max"])
)

resumen_estado.head()


Podemos regresar `state` a una columna:


In [ ]:
resumen_estado = resumen_estado.reset_index()
resumen_estado.head()


##### Nombrar los resultados

También podemos decidir cómo queremos llamar a cada nueva columna.

La sintaxis general es:

```python
nombre_nuevo = ("variable", "operación")
```


In [ ]:
resumen_estado = (
    df
    .groupby("state")
    .agg(
        observaciones=("value", "count"),
        promedio=("value", "mean"),
        mediana=("value", "median"),
        minimo=("value", "min"),
        maximo=("value", "max")
    )
    .reset_index()
)

resumen_estado.head()

#### Agrupar por más de una variable

También podemos definir grupos a partir de **más de una característica**.

Por ejemplo, calcular el promedio para cada combinación de `state` y `group`.


In [ ]:
df.groupby(
    ["group", "subgroup"]
)["value"].mean().head(50)


Podemos producir directamente un DataFrame resumido:


In [ ]:
promedio_estado_grupo = (
    df
    .groupby(["state", "group"])
    .agg(
        promedio=("value", "mean"),
        observaciones=("value", "count")
    )
    .reset_index()
)

promedio_estado_grupo.head(10)


También podemos agregar una tercera variable de agrupación:


In [ ]:
promedio_estado_grupo_fase = (
    df
    .groupby(["state", "group", "phase"])
    .agg(
        promedio=("value", "mean"),
        observaciones=("value", "count")
    )
    .reset_index()
)

promedio_estado_grupo_fase.head()


### Agrupar y ordenar

Podemos combinar varias operaciones. Por ejemplo:

> ¿Qué estados tienen el promedio más alto?


In [ ]:
(
    df
    .groupby("state")
    .agg(
        promedio=("value", "mean")
    )
    .reset_index()
    .sort_values(
        "promedio",
        ascending=False
    )
    .head(10)
)


### Combinar bases de datos

En la práctica, la información rara vez está contenida en una sola tabla.

Por ejemplo, podemos tener una tabla con observaciones y otra con información adicional sobre cada estado.

La variable que permite relacionarlas se conoce como **llave** o **identificador**.


#### Crear una segunda tabla

Construiremos una tabla pequeña con la región de algunos estados para practicar los joins.


In [ ]:
import pandas as pd

regiones = pd.DataFrame({
    "state": [
        "Vermont",
        "Wyoming",
        "Hawaii",
        "North Dakota",
        "California"
    ],
    "region": [
        "Northeast",
        "West",
        "West",
        "Midwest",
        "West"
    ]
})

regiones

Ahora tenemos dos tablas que comparten la variable `state`:


In [ ]:
df[["state", "value"]].head()


In [ ]:
regiones


#### Revisar la llave antes de combinar

Antes de hacer un `merge()`, conviene revisar si la llave contiene valores faltantes o duplicados.


In [ ]:
regiones["state"].isna().sum()

In [ ]:
regiones["state"].duplicated().sum()


In [ ]:
regiones["state"].unique()


#### `merge()`

Podemos combinar ambas tablas utilizando `merge()`.


In [ ]:
df.merge(
    regiones,
    on="state"
).head()


Una forma equivalente es usar `pd.merge()`:


In [ ]:
pd.merge(
    df,
    regiones,
    on="state"
).head()


### Tipos de joins

El argumento `how` determina qué observaciones queremos conservar.

- `inner`: solo coincidencias.
- `left`: todo lo de la tabla izquierda.
- `right`: todo lo de la tabla derecha.
- `outer`: todo lo de ambas tablas.


#### `inner join`

Conserva únicamente las observaciones que aparecen en **ambas tablas**.


In [ ]:
df_inner = df.merge(
    regiones,
    on="state",
    how="inner"
)

df_inner.head()


In [ ]:
df.shape, df_inner.shape


#### `left join`

Conserva **todas las observaciones de la tabla izquierda** y agrega información de la tabla derecha cuando existe una correspondencia.


In [ ]:
df_left = df.merge(
    regiones,
    on="state",
    how="left"
)

df_left.head()


Si un estado no aparece en `regiones`, la variable `region` tendrá un valor faltante (`NaN`).


In [ ]:
df_left[df_left["region"].isna()].head()


#### `right join`

Conserva todas las observaciones de la tabla de la derecha.


In [ ]:
df_right = df.merge(
    regiones,
    on="state",
    how="right"
)

df_right.head()


#### `outer join`

Conserva las observaciones de **ambas tablas**, aunque no tengan correspondencia.


In [ ]:
df_outer = df.merge(
    regiones,
    on="state",
    how="outer"
)

df_outer.head()


#### Identificar observaciones que no hicieron match

Hacer un `merge()` no garantiza que todas las observaciones hayan encontrado correspondencia.

Una primera estrategia es buscar valores faltantes después de un `left join`.


In [ ]:
df_left.loc[
    df_left["region"].isna(),
    "state"
].unique()


#### `indicator=True`

Pandas puede crear automáticamente una columna que indique de dónde provino cada observación.


In [ ]:
df_revision = df.merge(
    regiones,
    on="state",
    how="outer",
    indicator=True
)

df_revision.head()


La columna `_merge` puede tomar tres valores:

- `both`: encontró correspondencia.
- `left_only`: solo estaba en la tabla izquierda.
- `right_only`: solo estaba en la tabla derecha.


In [ ]:
df_revision["_merge"].value_counts()


Podemos inspeccionar los estados de `df` que no encontraron correspondencia:


In [ ]:
df_revision.loc[
    df_revision["_merge"] == "left_only",
    "state"
].unique()


Y los estados que aparecen únicamente en la tabla `regiones`:


In [ ]:
df_revision.loc[
    df_revision["_merge"] == "right_only",
    "state"
].unique()


#### Un problema muy común: llaves inconsistentes

Dos valores pueden parecer iguales para una persona, pero no necesariamente para Python.

Por ejemplo, diferencias en mayúsculas, minúsculas o espacios pueden hacer que un `merge()` falle.


In [ ]:
regiones_error = pd.DataFrame({
    "state": [
        "Vermont",
        "wyoming",
        "Hawaii ",
        "North Dakota"
    ],
    "region": [
        "Northeast",
        "West",
        "West",
        "Midwest"
    ]
})

regiones_error


In [ ]:
revision = df.merge(
    regiones_error,
    on="state",
    how="left",
    indicator=True
)

revision["_merge"].value_counts()


Podemos revisar qué estados no hicieron match:


In [ ]:
revision.loc[
    revision["_merge"] == "left_only",
    "state"
].unique()


Observa que estas comparaciones son falsas:


In [ ]:
"Wyoming" == "wyoming"


In [ ]:
"Hawaii" == "Hawaii "
